# PCA, Dimensionality Reduction & Clustering

**Author:** Dayton Wickerd  
**Date:** Spring 2026  

---

This notebook explores principal component analysis (PCA) and clustering techniques through a series of progressively complex datasets. It begins by generating synthetic bivariate normal data, applies rotation and scaling transformations, and examines how PCA recovers the principal axes. It then applies PCA-based dimensionality reduction and k-means and DBSCAN clustering to the Boston Housing and California Housing datasets, interpreting variance structure, cluster geometry, and geographic patterns.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import fetch_california_housing

np.random.seed(42)
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['axes.titlesize'] = 13
matplotlib.rcParams['axes.labelsize'] = 11
os.makedirs('figures', exist_ok=True)


---
# Section 1: Bivariate Normal Data Generation


## 1.1 Generate and Visualize Bivariate Normal Data

We draw 1,000 samples from a 2D standard normal distribution with mean **μ = [0, 0]** and identity covariance matrix **Σ = I**. Because the covariance matrix is the identity, the two dimensions are uncorrelated and have equal variance, so we expect the scatter to form a circular, isotropic cloud.


In [ ]:
mu = np.array([0, 0])
Sigma = np.array([[1, 0], [0, 1]])
X1, X2 = np.random.multivariate_normal(mu, Sigma, 1000).T
D = np.array([X1, X2]).T

plt.figure(figsize=(6, 6))
plt.scatter(X1, X2, color='steelblue', alpha=0.4, s=10)
plt.xlabel('X1')
plt.ylabel('X2')
plt.title('Bivariate Normal Data (mu=[0,0], Sigma=I)')
plt.axis('equal')
plt.tight_layout()
plt.savefig('figures/bivariate_normal.png', bbox_inches='tight')
plt.show()


### Interpretation

The scatter plot shows a roughly circular cloud centered near the origin, consistent with independent, equal-variance draws along both axes. There is no preferred direction of spread -- the cloud has no visible elongation or tilt -- which reflects the fact that X1 and X2 are uncorrelated (off-diagonal covariance is zero) and have identical variance (both equal to 1). Any apparent asymmetry is due to sampling noise from the finite 1,000-point sample.


---
# Section 2: Linear Transformations (Rotation & Scaling)


## 2.1 Define the Transformation

We apply a composed linear transformation to D: first **scale** by factors of 3 (along X1) and 4 (along X2) using a diagonal scaling matrix S, then **rotate** by 45 degrees counter-clockwise using a rotation matrix R. The combined transformation matrix is RS = R @ S, and the transformed dataset is RSD = D @ (RS)^T.


In [ ]:
# Scaling matrix: stretch by factor 3 along X1, factor 4 along X2
S = np.array([[3, 0], [0, 4]])

# Rotation matrix: 45-degree (pi/4 radian) counter-clockwise rotation
R = np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
              [np.sin(np.pi/4),  np.cos(np.pi/4)]])

# Combined transformation: scale first, then rotate
RS = R @ S

# Apply to data: each row of D is a 2D point
RSD = D @ RS.T


## 2.2 Plot Original and Transformed Data

Plotting both datasets on the same axes lets us visually confirm the effect of the rotation-scaling transformation. We expect the original circular cloud to become an elongated, tilted ellipse.


In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(X1, X2, color='steelblue', alpha=0.4, s=10, label='Original D')
plt.scatter(RSD[:, 0], RSD[:, 1], color='tomato', alpha=0.4, s=10, label='Transformed RSD')
plt.xlabel('X1')
plt.ylabel('X2')
plt.title('Original vs. Rotation-Scaled Transformation')
plt.legend()
plt.axis('equal')
plt.tight_layout()
plt.savefig('figures/linear_transformation.png', bbox_inches='tight')
plt.show()


### Interpretation

The original circular cloud (blue) has been transformed into an elongated ellipse (red) tilted at roughly 45 degrees. The scaling step stretches the cloud unequally along the two axes -- factor 4 along X2 creates more spread than factor 3 along X1 -- and the rotation step then tilts the major axis diagonally. The RSD cloud is also substantially larger than D overall, because scaling amplifies variance. The two clouds share the same center (near the origin) because the mean of D is [0, 0] and linear transformations preserve the zero mean.


## 2.3 Covariance Matrix of the Transformed Data

We compute the sample covariance matrix of RSD. Theoretically, if D has covariance Sigma, then the transformed data RSD has covariance RS @ Sigma @ (RS)^T. With Sigma = I (identity), this simplifies to RS @ (RS)^T, which is a full matrix with nonzero off-diagonal entries because the rotation has mixed the two dimensions.


In [ ]:
Transformed_Cov = np.cov(RSD.T)
print('Covariance matrix of RSD:')
print(Transformed_Cov.round(4))


### Interpretation

The covariance matrix of RSD has nonzero off-diagonal entries, indicating that the two dimensions are now correlated. This correlation was introduced by the rotation step: before rotation, S had stretched D into an axis-aligned ellipse with a diagonal covariance matrix [[9, 0], [0, 16]]; the 45-degree rotation then mixed X1 and X2, producing the observed off-diagonal terms. Theoretically, for S = diag(3, 4) and a 45-degree rotation, both diagonal entries converge to (9 + 16)/2 = 12.5 and the off-diagonal entries to (16 - 9)/2 = 3.5 as the sample size grows.


## 2.4 Total Variance of the Transformed Data

Total variance is the trace of the covariance matrix -- the sum of the individual feature variances.


In [ ]:
total_var_RSD = np.trace(Transformed_Cov)
print(f'Total variance of RSD: {total_var_RSD:.4f}')


## 2.5 Total Variance of the Original Data

For comparison, we compute the total variance of the original dataset D.


In [ ]:
Cov_D = np.cov(D.T)
total_var_D = np.trace(Cov_D)
print(f'Total variance of D: {total_var_D:.4f}')


### Interpretation

The total variance of D is approximately 2 (the trace of the 2x2 identity covariance matrix, with each feature contributing variance ~1). The total variance of RSD is approximately 25 -- the scaling matrix S increases variance multiplicatively: the squared scale factors are 3^2 + 4^2 = 9 + 16 = 25. Rotation does **not** change total variance because it is an orthogonal transformation that preserves norms and inner products. Therefore, total variance is determined entirely by the scaling step, not the rotation.


---
# Section 3: PCA on Transformed Data


## 3.1 Apply PCA to RSD

PCA finds the orthogonal directions of maximum variance in the data. Applying it to RSD should recover axes aligned with the major and minor axes of the ellipse -- effectively undoing the rotation introduced by R and revealing the scaling structure of S.


In [ ]:
pca = PCA(n_components=2)
transformed = pca.fit_transform(RSD)
print(f'Explained variance ratio: {pca.explained_variance_ratio_.round(4)}')


## 3.2 Plot PCA-Transformed Data

We plot the data projected onto the first two principal components. The x-axis corresponds to PC1 (direction of maximum variance) and the y-axis to PC2 (orthogonal direction of next greatest variance).


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(transformed[:, 0], transformed[:, 1],
            color='steelblue', alpha=0.4, s=10)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA-Transformed RSD Data')
plt.axis('equal')
plt.tight_layout()
plt.savefig('figures/pca_transformed.png', bbox_inches='tight')
plt.show()


### Interpretation

The PCA scatter plot shows an axis-aligned ellipse -- the major axis is now horizontal (along PC1) and the minor axis is vertical (along PC2). PCA has recovered the scaling structure of S by rotating the data so that the direction of greatest variance aligns with the x-axis. The cloud is wider than it is tall, reflecting that the first principal component captures more variance than the second. This is the expected result: PCA effectively reverses the rotation applied by R, revealing the underlying axis-aligned stretch introduced by S.


## 3.3 Covariance Matrix of PCA-Transformed Data

The covariance matrix of the PCA output reveals whether the principal components are uncorrelated, and how much variance each one captures.


In [ ]:
cov_PCA_transformed = np.cov(transformed.T)
print('Covariance matrix of PCA-transformed data:')
print(cov_PCA_transformed.round(6))


### Interpretation

The covariance matrix of the PCA-transformed data is approximately **diagonal** -- the off-diagonal entries are extremely close to zero (on the order of 1e-13 or smaller, essentially floating-point rounding error). This is the defining property of PCA: by rotating the data to align with the eigenvectors of the covariance matrix, it **decorrelates** the features. Each diagonal entry is the variance captured by the corresponding principal component. In contrast, the covariance matrix of RSD had nonzero off-diagonal entries due to the mixing introduced by rotation. PCA undoes that mixing, making the components independent (in the linear/covariance sense).


## 3.4 Variance Fraction per Principal Component

We compute the fraction of total variance captured by each principal component. These fractions sum to 1 since we retained all 2 dimensions.


In [ ]:
total_var = np.trace(cov_PCA_transformed)
pc1_frac = cov_PCA_transformed[0, 0] / total_var
pc2_frac = cov_PCA_transformed[1, 1] / total_var
print(f'PC1 fraction of variance: {pc1_frac:.4f} ({pc1_frac:.1%})')
print(f'PC2 fraction of variance: {pc2_frac:.4f} ({pc2_frac:.1%})')


### Interpretation

PC1 captures approximately **64%** of the total variance and PC2 captures approximately **36%**. Theoretically, these fractions correspond to the squared scaling factors of S divided by their sum: 4^2 / (3^2 + 4^2) = 16/25 = 0.64 for PC1 and 3^2 / 25 = 9/25 = 0.36 for PC2. PCA has correctly identified the 45-degree rotation and aligned the principal axes with the directions that S stretched the data most. The larger scale factor (4) dominates PC1, the smaller (3) dominates PC2.


---
# Section 4: PCA & Clustering on Boston Housing Dataset


## 4.1 Load the Boston Housing Dataset

The Boston Housing dataset contains 506 observations across 13 features describing neighborhoods in the Boston metropolitan area. We load it from a local CSV file.


In [ ]:
Boston_df = pd.read_csv('data/Boston.csv', index_col=0)
print(f'Shape: {Boston_df.shape}')
print(f'Features: {list(Boston_df.columns)}')


## 4.2 Standardize and Project to 2D via PCA

Before applying PCA, the data is standard-normalized (zero mean, unit variance per feature). This is essential because the 13 Boston features have very different scales and units -- without normalization, features with large numeric ranges would dominate the principal components regardless of their actual information content.


In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(Boston_df)

boston_pca = PCA(n_components=2)
boston_transformed = boston_pca.fit_transform(scaled_data)

plt.figure(figsize=(7, 6))
plt.scatter(boston_transformed[:, 0], boston_transformed[:, 1],
            color='steelblue', alpha=0.5, s=15)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('Boston Housing Data: 2D PCA Projection')
plt.tight_layout()
plt.savefig('figures/boston_pca_scatter.png', bbox_inches='tight')
plt.show()


### Interpretation

The 2D PCA projection shows the 506 Boston neighborhoods distributed along both principal components, with noticeably more spread along PC1 than PC2. There is a roughly elongated distribution with what appears to be two loosely separated regions -- a dense central cluster and a sparser high-PC1 tail. This visual separation hints at underlying structure in the housing data (likely separating high-value from low-value neighborhoods), which the clustering analysis below investigates more formally.


## 4.3 Variance Explained by Each Component (Scree Plot)

We fit PCA with all 13 components and plot the fraction of variance explained by each one. This scree plot guides how many components are needed to retain a given amount of information.


In [ ]:
full_boston_pca = PCA()
full_boston_pca.fit(scaled_data)

cumvar = np.cumsum(full_boston_pca.explained_variance_ratio_)
components = range(1, len(full_boston_pca.explained_variance_ratio_) + 1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(components, full_boston_pca.explained_variance_ratio_,
       color='steelblue', alpha=0.7, label='Per-component variance')
ax.plot(components, cumvar, color='tomato', marker='o',
        linestyle='--', label='Cumulative variance')
ax.axhline(0.9, color='gray', linestyle=':', linewidth=1, label='90% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Fraction of Variance Explained')
ax.set_title('PCA Variance Explained -- Boston Housing Dataset')
ax.set_xticks(list(components))
ax.legend()
plt.tight_layout()
plt.savefig('figures/boston_variance_explained.png', bbox_inches='tight')
plt.show()


### Interpretation

The scree plot shows a steep initial decline: the first few components each explain a substantial fraction of the variance, while subsequent components contribute progressively less. This concave-down shape -- often called an 'elbow curve' -- is typical of real-world datasets where a handful of latent factors drive most of the variation. The cumulative curve (red dashes) crosses the 90% threshold at 7 components, confirming that a large portion of the data's structure can be captured with roughly half the original dimensionality.


## 4.4 Minimum Components to Capture 90% of Variance

We compute the cumulative explained variance ratio to find the minimum number of principal components needed to retain at least 90% of the total variance.


In [ ]:
cumulative = np.cumsum(full_boston_pca.explained_variance_ratio_)
for k, v in enumerate(cumulative, start=1):
    print(f'PC{k:2d}: cumulative variance = {v:.4f} ({v:.1%})')


### Interpretation

**7 principal components** are needed to capture at least 90% of the total variance in the normalized Boston dataset. This means that 7 linear combinations of the original 13 features retain 90% of the information present in the full dataset. Reducing from 13 to 7 dimensions yields a nearly 50% reduction in dimensionality while sacrificing less than 10% of the variance -- a strong compression with minimal information loss. This also suggests that the 13 Boston features are not independent; they share substantial co-variation that PCA consolidates into fewer components.


## 4.5 Fraction of Variance Captured by the First 2 Components

The 2D projection used for visualization retains only the first 2 principal components. We compute the fraction of total variance this projection preserves.


In [ ]:
two_comp_var = cumulative[1]
print(f'Variance captured by PC1 + PC2: {two_comp_var:.4f} ({two_comp_var:.1%})')


### Interpretation

The first two principal components together capture approximately **60.9%** of the total variance in the normalized Boston dataset. While this is a substantial fraction for just 2 dimensions, nearly 40% of the information is still lost in the 2D scatter plot. This trade-off is inherent to PCA-based visualization: projecting high-dimensional data to 2D for human interpretation always involves some compression. The 60.9% retention is meaningful -- enough to reveal real structure -- but interpreting the scatter plot alone would miss nuances captured by the remaining 5 components needed to reach 90%.


## 4.6 K-Means Clustering (k=2) on PCA-Projected Boston Data

K-means is applied to the 2D PCA projection to partition the Boston neighborhoods into two clusters. K-means assumes spherical, equal-size clusters and minimizes within-cluster variance, so it works well when clusters are roughly convex and similarly sized.


In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(boston_transformed)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(boston_transformed[:, 0], boston_transformed[:, 1],
                     c=kmeans.labels_, cmap='Set1', alpha=0.6, s=20)
ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           color='black', marker='X', s=200, zorder=5, label='Centroids')
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_title('K-Means Clustering (k=2) -- Boston PCA Projection')
ax.legend()
plt.colorbar(scatter, ax=ax, label='Cluster')
plt.tight_layout()
plt.savefig('figures/boston_kmeans.png', bbox_inches='tight')
plt.show()


### Interpretation

K-means splits the Boston neighborhoods into two clusters separated primarily along PC1. The boundary runs roughly vertically in PCA space, with the left cluster (negative PC1) and the right cluster (positive PC1) reflecting the dominant axis of variation in the data. Because PC1 captures the most variance, this split likely separates neighborhoods along the most influential combined feature dimension -- possibly distinguishing lower-crime, higher-value areas from higher-crime, lower-value ones. The cluster centroids (black X markers) sit near the center of mass of each group. The two clusters appear roughly symmetric, consistent with k-means' assumption of equal-sized, spherical groups.


## 4.7 DBSCAN Clustering on PCA-Projected Boston Data

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) clusters points based on local density rather than distance to centroids. It requires no pre-specified number of clusters and can label sparse points as noise (label -1).


In [ ]:
dbscan = DBSCAN()
dbscan.fit(boston_transformed)

unique_labels = np.unique(dbscan.labels_)
cmap = plt.cm.get_cmap('Set1', max(1, len(unique_labels) - (1 if -1 in unique_labels else 0)))
color_idx = 0

fig, ax = plt.subplots(figsize=(8, 6))
for label in sorted(unique_labels):
    mask = dbscan.labels_ == label
    if label == -1:
        ax.scatter(boston_transformed[mask, 0], boston_transformed[mask, 1],
                   color='lightgray', alpha=0.6, s=20, label='Noise')
    else:
        ax.scatter(boston_transformed[mask, 0], boston_transformed[mask, 1],
                   color=cmap(color_idx), alpha=0.6, s=20, label=f'Cluster {label}')
        color_idx += 1
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_title('DBSCAN Clustering -- Boston PCA Projection')
ax.legend()
plt.tight_layout()
plt.savefig('figures/boston_dbscan.png', bbox_inches='tight')
plt.show()
print(f'Unique labels: {unique_labels}')
print(f'Noise points: {np.sum(dbscan.labels_ == -1)}')


### DBSCAN vs K-Means: Comparison

**K-means** partitions every point into one of exactly two clusters, drawing a clean boundary through the middle of the PCA space. It is a hard partition with no concept of noise -- every neighborhood must belong to a cluster even if it is an outlier.

**DBSCAN** identifies dense regions and labels sparse or isolated points as noise (gray points). With default parameters (eps=0.5, min_samples=5), DBSCAN typically identifies the dense core of the Boston PCA data as one or two clusters while flagging the high-PC1 outliers -- likely the extreme suburban or industrial neighborhoods -- as noise. This is a more nuanced view of the data structure: rather than forcing every point into a group, it distinguishes genuine cluster membership from outlier status.

The two methods answer different questions: k-means asks 'how do I split this population into groups?', while DBSCAN asks 'where are the natural dense neighborhoods in this space, and which points do not belong to any of them?'


---
# Section 5: DBSCAN Clustering on California Housing Dataset


## 5.1 Load the California Housing Dataset

The California Housing dataset contains 20,640 housing records across 8 features derived from the 1990 California census. Crucially, it includes latitude and longitude coordinates, making it possible to visualize clusters as geographic regions.


In [ ]:
california_housing = fetch_california_housing(as_frame=True)
cal_df = california_housing.data
print(f'Shape: {cal_df.shape}')
print(f'Features: {list(cal_df.columns)}')


## 5.2 Standardize Features

DBSCAN uses distance-based density estimation, so all features must be on the same scale before clustering. We apply standard normalization (zero mean, unit variance) to the 8 housing features. The target column (median house value) is excluded so that clustering reflects feature structure, not the label.


In [ ]:
cal_scaler = StandardScaler()
cal_scaled = cal_scaler.fit_transform(cal_df)
print(f'Scaled data shape: {cal_scaled.shape}')


## 5.3 DBSCAN Clustering and Geographic Visualization

We run DBSCAN on the standardized feature matrix. We then plot the results using longitude (x-axis) and latitude (y-axis) from the original unscaled data, with each point colored by its DBSCAN cluster label. This allows us to see whether the density-based clusters correspond to recognizable geographic regions in California.


In [ ]:
dbscan_cal = DBSCAN()
dbscan_cal.fit(cal_scaled)

unique_labels_cal = np.unique(dbscan_cal.labels_)
n_clusters = len(unique_labels_cal) - (1 if -1 in unique_labels_cal else 0)
print(f'Number of clusters found: {n_clusters}')
print(f'Noise points: {np.sum(dbscan_cal.labels_ == -1)}')

cmap_cal = plt.cm.get_cmap('tab10', max(1, n_clusters))
color_idx = 0

fig, ax = plt.subplots(figsize=(10, 8))
for label in sorted(unique_labels_cal):
    mask = dbscan_cal.labels_ == label
    if label == -1:
        ax.scatter(cal_df['Longitude'][mask], cal_df['Latitude'][mask],
                   color='lightgray', alpha=0.2, s=2, label='Noise')
    else:
        ax.scatter(cal_df['Longitude'][mask], cal_df['Latitude'][mask],
                   color=cmap_cal(color_idx), alpha=0.5, s=3,
                   label=f'Cluster {label}')
        color_idx += 1
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('DBSCAN Geographic Clusters -- California Housing Dataset')
ax.legend(loc='upper right', markerscale=4, fontsize=9)
plt.tight_layout()
plt.savefig('figures/california_dbscan.png', bbox_inches='tight')
plt.show()


### Interpretation

Plotting DBSCAN's cluster labels on a longitude-latitude map reveals that the algorithm's density-based clusters correspond directly to **geographic regions** of California. Dense urban areas -- most visibly the **San Francisco Bay Area** (upper left), **Los Angeles** (lower center), and **San Diego** (lower right) -- emerge as distinct clusters because they contain many housing records with similar feature profiles in close spatial proximity.

Rural and sparsely populated areas are predominantly labeled as **noise** (gray points), because isolated housing records do not meet the density threshold required to form a cluster under the default DBSCAN parameters.

This result highlights a key advantage of DBSCAN over k-means for spatial or irregularly shaped data: DBSCAN does not force every point into a cluster and can discover clusters of arbitrary shape. The geographic boundaries of the clusters follow natural density contours rather than circular regions, making them more interpretable in a real-world context. The strong correspondence between clusters and known metropolitan regions also suggests that geographic proximity and feature similarity are tightly coupled in this dataset -- as expected, since location strongly drives housing prices, income levels, and population density.
